# ⚡ Duino API — Google Colab

**Cloud-agnostic hyperscale AI inference — Gemma 4 · Streaming · React UI**

> 💡 **Before running:** `Runtime → Change runtime type → T4 GPU`  
> 🔑 **HF Token:** Add `HF_TOKEN` to Colab Secrets (🔑 icon in sidebar)  
> 🌐 **Public HTTPS links** are auto-generated for both API and UI

---

## 📦 Cell 1 — Clone & Install
_Run once (~2 min)._

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jalalmansour/Duino_API.git'
REPO_DIR = '/content/Duino_API'

if not os.path.exists(REPO_DIR):
    print('Cloning Duino API...')
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Pulling latest...')
    !git -C {REPO_DIR} pull --quiet

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print('Upgrading setuptools...')
!pip install --upgrade setuptools wheel pip -q
print('Installing core requirements...')
!pip install -r /content/Duino_API/requirements.txt -q
print('Installing duino-api...')
r = subprocess.run(['pip','install','-e',REPO_DIR,'-q','--no-build-isolation'],
                   capture_output=True, text=True)
if r.returncode != 0:
    subprocess.run(['pip','install',REPO_DIR,'-q'], check=True)
print('\n✅ Setup complete!')

## 🔑 Cell 2 — HuggingFace Token

In [ ]:
import os
# Add HF_TOKEN via 🔑 Colab Secrets sidebar (recommended)
# os.environ['HF_TOKEN']        = 'hf_...'  # or set here temporarily
# os.environ['NGROK_AUTHTOKEN'] = '...'
print(f"HF_TOKEN        : {'✅ set' if os.environ.get('HF_TOKEN')        else '⚠️  NOT SET'}")
print(f"NGROK_AUTHTOKEN : {'✅ set' if os.environ.get('NGROK_AUTHTOKEN') else '⚠️  not set (Colab proxy used — OK)'}")

## 🚀 Cell 3 — Start Platform
_Starts API + UI. Prints public shareable HTTPS URLs._

In [ ]:
import sys, os, time
REPO_DIR = '/content/Duino_API'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from studio.backend.colab import start

result = start(
    model    = 'gemma-4-2b',
    api_port = 8000,
    ui_port  = 3000,
    expose   = True,
)

api_url    = result['api_url']
ui_url     = result['ui_url']
api_key    = result['api_key']
embed_html = result['embed_html']

print('\n' + '═'*60)
print('  🌐 PUBLIC LINKS — share these with anyone')
print('═'*60)
print(f'  📡 API      : {api_url}')
print(f'  📖 Docs     : {api_url}/docs')
print(f'  🎨 UI       : {ui_url}')
print(f'  🔑 API Key  : {api_key}')
print('═'*60)

## 🎨 Cell 4 — Embed UI & API Docs

In [ ]:
from google.colab import output as colab_output
print('🎨 React Chat UI (port 3000)')
colab_output.serve_kernel_port_as_iframe(3000, height=700, width='100%')

In [ ]:
from google.colab import output as colab_output
print('📖 API Docs / Swagger (port 8000)')
colab_output.serve_kernel_port_as_iframe(8000, height=900, width='100%')

## 🧪 Cell 5 — Test API

In [ ]:
import requests, json

# Single request
resp = requests.post(f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={'model':'gemma-4-2b',
          'messages':[{'role':'user','content':'What is the capital of France?'}],
          'max_tokens':64})
print('Answer:', resp.json()['choices'][0]['message']['content'])

# Streaming
print('\nStreaming:')
with requests.post(f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={'model':'gemma-4-2b',
          'messages':[{'role':'user','content':'Write a haiku about the ocean.'}],
          'max_tokens':64,'stream':True}, stream=True) as r:
    for line in r.iter_lines():
        if line:
            l = line.decode()
            if l.startswith('data: ') and l[6:] != '[DONE]':
                print(json.loads(l[6:])['choices'][0].get('delta',{}).get('content',''),
                      end='', flush=True)
print()

## 💬 Cell 6 — Multi-turn Chat

In [ ]:
import requests
sess = requests.post(f'{api_url}/v1/sessions',
    headers={'X-API-Key': api_key},
    json={'model_id': 'gemma-4-2b'}).json()
session_id = sess['session_id']

def chat(msg):
    return requests.post(f'{api_url}/v1/chat/completions',
        headers={'X-API-Key': api_key},
        json={'model':'gemma-4-2b',
              'messages':[{'role':'user','content':msg}],
              'session_id':session_id,'max_tokens':256}
    ).json()['choices'][0]['message']['content']

print('AI:', chat('My name is Jalal. Remember it.'))
print('AI:', chat('What is my name?'))

## ⚛️ Cell 7 — Vite / Next.js Inside Colab

In [ ]:
import subprocess, threading, time, os
from google.colab import output as colab_output

REACT_PORT = 4000
REACT_DIR  = '/content/my-react-app'
if not os.path.exists(REACT_DIR):
    !npm create vite@latest {REACT_DIR} -- --template react -y 2>/dev/null
    !npm install --prefix {REACT_DIR} --silent

def _vite():
    subprocess.run(['npm','run','dev','--','--host','0.0.0.0','--port',str(REACT_PORT)],
                   cwd=REACT_DIR)
threading.Thread(target=_vite, daemon=True).start()
time.sleep(6)
colab_output.serve_kernel_port_as_iframe(REACT_PORT, height=600, width='100%')

## 📊 Cell 8 — Health & Models

In [ ]:
import requests
health = requests.get(f'{api_url}/v1/health').json()
models = requests.get(f'{api_url}/v1/models', headers={'X-API-Key': api_key}).json()
print('📊 Health')
for k, v in health.items(): print(f'  {k:<24}: {v}')
print('\n🤖 Models')
for m in models.get('data', []): print(f"  - {m['id']}")
print(f'\n🌐 API  : {api_url}')
print(f'📖 Docs : {api_url}/docs')
print(f'🎨 UI   : {ui_url}')

## ♾️ Cell 9 — Keep Alive (NEVER STOPS)
_Keeps the session alive indefinitely. Uses JS anti-disconnect + API watchdog + exception recovery._

In [ ]:
import time, threading, requests
from IPython.display import display, Javascript
from google.colab import output as colab_output

# ── 1. Embed the UI (stays visible while keep-alive runs) ─────────────────────
print('═'*60)
print('  🌐 PUBLIC LINKS')
print('═'*60)
print(f'  📡 API  : {api_url}')
print(f'  📖 Docs : {api_url}/docs')
print(f'  🎨 UI   : {ui_url}')
print(f'  🔑 Key  : {api_key}')
print('═'*60)

colab_output.serve_kernel_port_as_iframe(3000, height=1200, width='100%')

# ── 2. JavaScript anti-disconnect ─────────────────────────────────────────────
# Simulates a click every 60s to prevent Colab's 90-min idle timeout
display(Javascript("""
function keepColabAlive() {
    try {
        // Click the "Connect" button if it exists (reconnects if disconnected)
        var connectBtn = document.querySelector('#top-toolbar > colab-connect-button');
        if (connectBtn) connectBtn.click();
        // Simulate activity to reset idle timer
        document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true}));
    } catch(e) {}
    setTimeout(keepColabAlive, 60000);  // every 60 seconds
}
keepColabAlive();
console.log('[Duino] Anti-disconnect watchdog started');
"""))

# ── 3. Python watchdog: pings API + keeps cell running ────────────────────────
_start_time = time.time()
_ping_count  = 0
_ping_errors = 0

def _elapsed():
    s = int(time.time() - _start_time)
    h, m = divmod(s, 3600)
    m, s = divmod(m, 60)
    return f'{h:02d}h {m:02d}m {s:02d}s'

print('\n[Keep-Alive] Started — this cell NEVER stops automatically.')
print('[Keep-Alive] Press STOP (■) only if you want to terminate.')
print()

PING_INTERVAL = 120   # seconds between API health pings
PRINT_EVERY   = 5     # print status every N pings

while True:  # Infinite loop — only user STOP can end this
    try:
        time.sleep(PING_INTERVAL)
        _ping_count += 1

        # Ping the API health endpoint to keep the server warm
        try:
            r = requests.get(f'{api_url}/v1/health', timeout=10)
            ok = r.status_code == 200
        except Exception:
            ok = False
            _ping_errors += 1

        # Print heartbeat every PRINT_EVERY pings
        if _ping_count % PRINT_EVERY == 0 or not ok:
            status = '✅' if ok else '⚠️ '
            print(f'[{_elapsed()}] {status} ping #{_ping_count} | '
                  f'errors: {_ping_errors} | '
                  f'API: {api_url}',
                  flush=True)

    except KeyboardInterrupt:
        # User pressed STOP — exit gracefully
        print(f'\n[Keep-Alive] Stopped after {_elapsed()} ({_ping_count} pings)')
        break
    except Exception as e:
        # Any other exception — log and continue (never crash)
        _ping_errors += 1
        print(f'[Keep-Alive] ⚠️  error (continuing): {e}', flush=True)
        time.sleep(10)  # brief pause before retrying
        continue